# Inference: anonymize faces and license plates with a trained segmentation model

Loads a model saved by `trainer/segmentation_acc.py` or `trainer/segmentation_opt.py` from `model_weights/<model id>`,
measures it on a test split from the data pipeline, shows full-image examples and close-ups, runs it on unlabeled
images (`data_pipeline/folder_to_unlabeled.py`), and anonymizes any image file.

Inference runs on the full image: upscale by the model's factor, run the model on sliding windows of its input size,
merge, and map the probabilities back to the original resolution. Pixels whose `P(not background)` is above `THRESHOLD` are blurred.

## 1. Settings

In [ ]:
WEIGHTS_DIR = "model_weights/v1_pp4av_seg3_mita_segmentation_vit_tiny"   # model_weights/<model id>
TEST_PATHS = ["data/v1/pp4av_segmentation_test"]                       # merged, like dataset_paths in a config
UNLABELED_PATHS = []       # e.g. ["data/v1/dashcam_unlabeled"] from data_pipeline/folder_to_unlabeled.py
N_UNLABELED = 6            # random unlabeled images shown in section 5
DEVICE = "cuda"
EVAL_LIMIT = None          # images used for the test metrics (None = all)
WINDOW_BATCH = 16          # sliding windows per forward pass

THRESHOLD = 0.5            # P(private) above which a pixel is masked
MASK_DILATE_PX = 3         # grow the mask slightly before blurring
BLUR_SIGMA = 8

In [ ]:
import json, random, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from models.mita_segmentation import MitaSegmentation
from trainer.common import load_merged_dataset
from trainer.segmentation import decode_row, evaluate, undecoded

model = MitaSegmentation.load(WEIGHTS_DIR, device=DEVICE)
state = json.loads((Path(WEIGHTS_DIR) / "train_state.json").read_text())
test_ds = load_merged_dataset(TEST_PATHS)
pd.Series({
    **{k: v for k, v in model.config.items()},
    "trained": f"epoch {state['epoch']}, step {state['step']}" if "epoch" in state else f"optuna trial {state['trial']}",
    "model input": f"{model.input_size}x{model.input_size} px, upscale {model.upscale:g}x",
    "test images": f"{len(test_ds)} from {sorted(set(test_ds['subset']))}",
}).to_frame("model")

## 2. Test metrics

Per-class Dice and IoU of the argmax prediction over all pixels of the test images (the same measure as the trainer's validation),
next to the validation metrics stored with the weights.

In [ ]:
eval_ds = test_ds if EVAL_LIMIT is None else test_ds.select(range(min(EVAL_LIMIT, len(test_ds))))
t = time.perf_counter()
test_metrics = evaluate(model, eval_ds, batch_size=WINDOW_BATCH)
print(f"{len(eval_ds)} images in {time.perf_counter() - t:.0f} s")
pd.DataFrame({"test": test_metrics, "valid (at training)": state.get("valid", {})}).round(4)

## 3. Full images: the test images with the most objects

In [ ]:
@torch.no_grad()
def private_prob(img_rgb):
    """RGB uint8 [H, W, 3] -> P(not background) float32 [H, W]."""
    return (1 - model.predict(img_rgb, batch_size=WINDOW_BATCH)[0]).cpu().numpy()

def anonymize(img_rgb, prob):
    mask = (prob > THRESHOLD).astype(np.uint8)
    if MASK_DILATE_PX > 0:
        k = 2 * MASK_DILATE_PX + 1
        mask = cv2.dilate(mask, np.ones((k, k), np.uint8))
    blurred = cv2.GaussianBlur(img_rgb, (0, 0), BLUR_SIGMA)
    return np.where(mask[..., None] == 1, blurred, img_rgb), mask

def objects(label_mask):
    """Connected regions of labelled (non-background) pixels -> boxes x1, y1, x2, y2."""
    n, _, stats, _ = cv2.connectedComponentsWithStats((label_mask > 0).astype(np.uint8), connectivity=8)
    return [(x, y, x + w, y + h) for x, y, w, h in stats[1:, :4]]

raw = undecoded(test_ds)
n_objects = [len(objects(cv2.imdecode(np.frombuffer(r["bytes"], np.uint8), cv2.IMREAD_UNCHANGED))) for r in raw["mask"]]
results = []
for i in np.argsort(n_objects)[::-1][:4]:
    row = raw[int(i)]
    img, label = decode_row(row)
    t = time.perf_counter(); prob = private_prob(img); dt = time.perf_counter() - t
    anon, mask = anonymize(img, prob)
    results.append((row, img, label, anon))

    fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))
    axes[0].imshow(img)
    for x1, y1, x2, y2 in objects(label):
        axes[0].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", lw=1.5))
    axes[0].set_title(f"{row['image_id']}: input + labelled regions (green)")
    axes[1].imshow(prob, cmap="magma", vmin=0, vmax=1); axes[1].set_title(f"P(private)   ({dt * 1000:.0f} ms)")
    axes[2].imshow(anon); axes[2].set_title("anonymized")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

## 4. Close-ups: labelled objects, original vs. anonymized

In [ ]:
pool = [(img, anon, b) for row, img, label, anon in results for b in objects(label)]
random.Random(0).shuffle(pool)
pool = pool[:12]
if pool:
    fig, axes = plt.subplots(2, len(pool), figsize=(1.6 * len(pool), 3.6), squeeze=False)
    for k, (img, anon, (x1, y1, x2, y2)) in enumerate(pool):
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        r = int(max(x2 - x1, y2 - y1) * 1.5 + 8)
        sl = (slice(max(0, int(cy) - r), int(cy) + r), slice(max(0, int(cx) - r), int(cx) + r))
        axes[0, k].imshow(img[sl], interpolation="nearest"); axes[1, k].imshow(anon[sl], interpolation="nearest")
        axes[0, k].set_title(f"{x2 - x1}x{y2 - y1}px", fontsize=8)
        axes[0, k].axis("off"); axes[1, k].axis("off")
    axes[0, 0].text(-0.15, 0.5, "original", transform=axes[0, 0].transAxes, rotation=90, va="center", ha="right")
    axes[1, 0].text(-0.15, 0.5, "anonymized", transform=axes[1, 0].transAxes, rotation=90, va="center", ha="right")
    plt.tight_layout(); plt.show()

## 5. Unlabeled images

Random images from the `UNLABELED_PATHS` datasets: no labels, so only the prediction and the anonymized result.

In [ ]:
if UNLABELED_PATHS:
    unlabeled = undecoded(load_merged_dataset(UNLABELED_PATHS).select_columns(["image_id", "image"]))
    picks = random.Random(0).sample(range(len(unlabeled)), min(N_UNLABELED, len(unlabeled)))
    for i in picks:
        row = unlabeled[i]
        img = cv2.cvtColor(cv2.imdecode(np.frombuffer(row["image"]["bytes"], np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        t = time.perf_counter(); prob = private_prob(img); dt = time.perf_counter() - t
        anon, mask = anonymize(img, prob)
        fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))
        axes[0].imshow(img); axes[0].set_title(f"{row['image_id']}  ({img.shape[1]}x{img.shape[0]})")
        axes[1].imshow(prob, cmap="magma", vmin=0, vmax=1); axes[1].set_title(f"P(private)   ({dt * 1000:.0f} ms)")
        axes[2].imshow(anon); axes[2].set_title(f"anonymized ({100 * mask.mean():.2f}% of pixels)")
        for ax in axes: ax.axis("off")
        plt.tight_layout(); plt.show()

## 6. Anonymize an image file

In [ ]:
IMAGE_PATH = None          # e.g. "/path/to/street.jpg"
OUTPUT_PATH = None         # e.g. "/path/to/street_anonymized.jpg"; None = only show it

if IMAGE_PATH is not None:
    img = cv2.cvtColor(cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    anon, mask = anonymize(img, private_prob(img))
    if OUTPUT_PATH is not None:
        cv2.imwrite(OUTPUT_PATH, cv2.cvtColor(anon, cv2.COLOR_RGB2BGR))
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    axes[0].imshow(img); axes[0].set_title("input"); axes[1].imshow(anon); axes[1].set_title(f"anonymized ({100 * mask.mean():.2f}% of pixels)")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()